# Heart Failure Demo

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

DATA_PATH = Path('data/heart_failure_clinical_records_dataset.csv')
df = pd.read_csv(DATA_PATH)

clean_df = df.drop_duplicates().copy()
if clean_df.isnull().sum().sum() > 0:
    clean_df = clean_df.dropna().reset_index(drop=True)

print(f'Dataset loaded: {clean_df.shape[0]} rows, {clean_df.shape[1]} columns')

Dataset loaded: 299 rows, 13 columns


In [2]:
feature_columns = [column for column in clean_df.columns if column != 'DEATH_EVENT']
continuous_features = [
    'age',
    'creatinine_phosphokinase',
    'ejection_fraction',
    'platelets',
    'serum_creatinine',
    'serum_sodium',
    'time'
]

def train_test_split_from_scratch(X, y, test_size=0.2, random_state=42):
    rng = np.random.default_rng(random_state)
    indices = np.arange(X.shape[0])
    rng.shuffle(indices)
    test_count = int(np.ceil(X.shape[0] * test_size))
    test_idx = indices[:test_count]
    train_idx = indices[test_count:]
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

def fit_standardizer(train_df, columns):
    means = train_df[columns].mean()
    stds = train_df[columns].std(ddof=0).replace(0, 1.0)
    return means, stds

def apply_standardizer(frame, columns, means, stds):
    transformed = frame.copy()
    transformed[columns] = (transformed[columns] - means) / stds
    return transformed

X_df = clean_df[feature_columns].copy()
y = clean_df['DEATH_EVENT'].to_numpy(dtype=float)

X_train_raw, X_test_raw, y_train, y_test = train_test_split_from_scratch(
    X_df.to_numpy(dtype=float),
    y,
    test_size=0.2,
    random_state=42
)

X_train_df = pd.DataFrame(X_train_raw, columns=feature_columns)
X_test_df = pd.DataFrame(X_test_raw, columns=feature_columns)

train_means, train_stds = fit_standardizer(X_train_df, continuous_features)
X_train_processed = apply_standardizer(X_train_df, continuous_features, train_means, train_stds)
X_test_processed = apply_standardizer(X_test_df, continuous_features, train_means, train_stds)

In [3]:
# ChatGPT was used to outline the class skeleton and function descriptions, 
# but the implementation and debugging were done by hand.
class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.05, epochs=4000):
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = 0.0

    @staticmethod
    def sigmoid(z):
        z = np.clip(z, -500, 500)
        return 1.0 / (1.0 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features, dtype=float)
        self.bias = 0.0

        for _ in range(self.epochs):
            linear_output = np.dot(X, self.weights) + self.bias
            predictions = self.sigmoid(linear_output)
            dw = np.dot(X.T, (predictions - y)) / n_samples
            db = np.mean(predictions - y)
            self.weights -= self.learning_rate * dw
            self.bias -= self.learning_rate * db

        return self

    def predict_proba(self, X):
        linear_output = np.dot(X, self.weights) + self.bias
        return self.sigmoid(linear_output)

    def predict(self, X, threshold=0.5):
        probabilities = self.predict_proba(X)
        return (probabilities >= threshold).astype(int)

X_train = X_train_processed.to_numpy(dtype=float)
model = LogisticRegressionScratch(learning_rate=0.05, epochs=4000)
model.fit(X_train, y_train)

print('Model training complete.')

Model training complete.


In [ ]:
def prepare_single_patient(patient_dict, feature_columns, continuous_features, means, stds):
    patient_df = pd.DataFrame([patient_dict], columns=feature_columns)
    patient_df = apply_standardizer(patient_df, continuous_features, means, stds)
    return patient_df.to_numpy(dtype=float)

def predict_patient_risk(patient_dict, threshold=0.5):
    patient_array = prepare_single_patient(
        patient_dict,
        feature_columns,
        continuous_features,
        train_means,
        train_stds
    )
    probability = float(model.predict_proba(patient_array)[0])
    prediction = int(probability >= threshold)
    return probability, prediction
# Change the demo characteristics to try out the model
demo_patient = {
    'age': 15.0,
    'anaemia': 0,
    'creatinine_phosphokinase': 250.0,
    'diabetes': 1,
    'ejection_fraction': 35.0,
    'high_blood_pressure': 1,
    'platelets': 250000.0,
    'serum_creatinine': 1.3,
    'serum_sodium': 137.0,
    'sex': 1,
    'smoking': 0,
    'time': 120.0
}

probability, prediction = predict_patient_risk(demo_patient)

print('Demo patient:')
print(pd.DataFrame([demo_patient]).to_string(index=False))
print(f'Predicted probability of death event: {probability:.4f}')

Demo patient:
 age  anaemia  creatinine_phosphokinase  diabetes  ejection_fraction  high_blood_pressure  platelets  serum_creatinine  serum_sodium  sex  smoking  time
15.0        0                     250.0         1               35.0                    1   250000.0               1.3         137.0    1        0 120.0
Predicted probability of death event: 0.0333
Predicted class at threshold 0.50: 0
